# Romania 2001: Bucharest gymnasium applicant acquisition

**B = Bucharest.** Its saved Ministry origin-school directory lists **289 gymnasium applicant webpages**. The 297 program-specific placement reports have already been recovered and reconciled. This notebook obtains the originating-gymnasium applicant records needed for peer-group reconstruction; it does not repeat the placement recovery or run an outcome analysis.

**Run Cells 1 → 2 → 3 → 4.** Cell 1 already has `COUNTIES = ("B",)` and `RUN_DOWNLOAD = True`. The trailing comma makes this a one-item tuple. Cell 3 previews the saved-page count without network access. **Cell 4 is the download run you start.** Cell 5 reads saved progress after completion or interruption.

Use the `sports_net` kernel. Keep other Wayback acquisition notebooks stopped while Cell 4 runs. Each verified page is saved immediately in the private Desktop cache; rerunning skips saved pages and retries unresolved ones. Bucharest has its own progress CSV, so this run does not replace the Arad/Sibiu progress file.

Downloading all gymnasium pages is one acquisition step. The county applicant/unassigned lists and student-to-placement matches must still be checked before Bucharest can enter the analysis. No missing source is treated as proof of failure or permanent absence.


In [ ]:
# CELL 1 — SETTINGS YOU CONTROL
# Re-run this cell after changing a setting. Downloads start only when you run Cell 4.
from pathlib import Path

REPO_ROOT = Path("/Users/charleslevine/Library/CloudStorage/Dropbox/1-Documents/00- Dissertation/0-Next_Chapter/Code_and_Data/New SQL and PY Code/Cursor Workspace PDE")
ANALYSIS_ROOT = REPO_ROOT / "3-Master_Plan/VECTOR_work/new_VECTOR_work/education_analysis"
COUNTIES = ("B",)  # Bucharest only; keep the trailing comma.
RUN_DOWNLOAD = True

# Each verified page is checkpointed immediately. These caps bound one run;
# restart the notebook later to resume, rather than starting from page one.
MAX_NEW_PAGES = 300
MAX_HTTP_ATTEMPTS = 1000
MAX_HOURS = 12.0

# The worker adapts its pace, reports each request/wait, and responds to
# server pressure. These values preserve the previous successful policy.
MIN_INTERVAL_SECONDS = 22.0
MAX_INTERVAL_SECONDS = 50.0
FIRST_429_COOLDOWN_SECONDS = 600
SECOND_429_COOLDOWN_SECONDS = 1500
SERVER_ERROR_WAIT_SECONDS = 10.0

STATUS_FILE = ANALYSIS_ROOT / "outputs/romania_2001_bucharest_expansion_20261005/gymnasium_acquisition_status.csv"
print("Counties:", ", ".join(COUNTIES))
print("Download enabled:", RUN_DOWNLOAD)
print("Per-request spacing:", MIN_INTERVAL_SECONDS, "to", MAX_INTERVAL_SECONDS, "seconds")
print("Name-free status file:", STATUS_FILE)
print("No network request has been made by this cell.")

## Cell 2: load the downloader

This imports and reloads the shared Python downloader, including support for `B`. It makes no network request. Raw school webpages remain under the private Desktop cache; only aggregate progress goes into the repository.


In [ ]:
# CELL 2 — IMPORT ONLY; NO NETWORK REQUEST
import importlib
import sys

CODE_DIR = ANALYSIS_ROOT / "code"
if not CODE_DIR.is_dir():
    raise FileNotFoundError(f"Code folder not found: {CODE_DIR}")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
worker = importlib.import_module("EDUCATION_20261002_romania_three_county_gymnasium_acquisition")
worker = importlib.reload(worker)
if not set(COUNTIES).issubset(set(worker.COUNTIES)):
    raise ValueError(f"Unsupported county in {COUNTIES}; worker supports {worker.COUNTIES}")
print("Loaded source-only worker:", worker.__file__)
print("Private saved-page folder:", worker.PAGES)
print("No Wayback request has been made.")

## Cell 3: preview what remains

This reads the saved Ministry directory and checks existing local school-page checkpoints. It makes no network request and writes no files. Before the first Bucharest run, expect **0/289** verified school webpages. Later previews count whatever has already been saved.


In [ ]:
# CELL 3 — READ-ONLY PREVIEW; NO NETWORK REQUEST
worker.main([
    "--counties", *COUNTIES,
    "--status-file", str(STATUS_FILE),
])

## Cell 4: start or resume the download

Run this after Cells 1–3. The settings already permit downloading, but **nothing downloads until this cell runs**. Each request prints its URL, each new saved page prints its row count, and every wait reports its reason and expected resume time.

The notebook retains the established 22–50 second adaptive pacing, 600- then 1,500-second pauses after repeated HTTP 429 responses, and at least ten seconds after HTTP 500/503. A third consecutive 429 stops the pass with checkpoints preserved. The full 429 response and exact URL are printed. Runtime/request limits also preserve everything; rerun this cell later to continue.

Keep this as the only Wayback downloader running. Interrupting the cell leaves completed checkpoints available.


In [ ]:
# CELL 4 — RESEARCHER-CONTROLLED SOURCE ACQUISITION
if not RUN_DOWNLOAD:
    print("Download is OFF. Change RUN_DOWNLOAD to True in Cell 1 and rerun Cell 1 first.")
else:
    run_args = [
        "--run", "--counties", *COUNTIES,
        "--status-file", str(STATUS_FILE),
        "--max-new-pages", str(MAX_NEW_PAGES),
        "--max-http-attempts", str(MAX_HTTP_ATTEMPTS),
        "--max-hours", str(MAX_HOURS),
        "--min-interval-seconds", str(MIN_INTERVAL_SECONDS),
        "--max-interval-seconds", str(MAX_INTERVAL_SECONDS),
        "--first-429-cooldown-seconds", str(FIRST_429_COOLDOWN_SECONDS),
        "--second-429-cooldown-seconds", str(SECOND_429_COOLDOWN_SECONDS),
        "--server-error-wait-seconds", str(SERVER_ERROR_WAIT_SECONDS),
    ]
    print("Starting SOURCE-ONLY acquisition for:", ", ".join(COUNTIES), flush=True)
    print("Expected directory-listed school webpages: B (Bucharest), 289.", flush=True)
    print("Progress will be checkpointed after every requested school.", flush=True)
    worker.main(run_args)

## Cell 5: inspect the saved progress

This cell makes no network request. It shows how many Bucharest school webpages are saved and how many remain unresolved. An unresolved webpage is not a school with zero applicants. After all 289 verify, the next step is offline reconciliation of applicants, placements, and remaining source gaps.


In [ ]:
# CELL 5 — LOCAL PROGRESS CHECK; NO NETWORK REQUEST
import pandas as pd
if not STATUS_FILE.exists():
    print("No Bucharest status file yet. Cell 3 is preview only; Cell 4 writes this file.")
else:
    progress = pd.read_csv(STATUS_FILE, dtype={"gymnasium_code": str}).fillna("")
    progress = progress.loc[progress["county"].isin(COUNTIES)].copy()
    totals = progress.groupby(["county", "status"], dropna=False).size().rename("school_webpages").reset_index()
    display(totals)
    for county in COUNTIES:
        local = progress.loc[progress["county"] == county]
        verified = int(local["status"].eq("saved_verified").sum())
        print(f"{county}: {verified}/{len(local)} gymnasium webpages verified")
    unresolved = progress.loc[progress["status"].str.startswith("unresolved")]
    print("Unresolved school webpages:", len(unresolved))
    if not unresolved.empty:
        display(unresolved[["county", "gymnasium_code", "status", "last_http_status"]])
    print("Status file:", STATUS_FILE)